# 02 · NYUDv2 and MultiCue stores  (step 12 preparation)
**Accelerator: None (CPU).** Run with **Save Version → Save & Run All (Commit)**; you can close the laptop. Re-running continues where the last run stopped, on any account.


Attach the downloaded datasets, **inspect their layout first**, write list files `image gt1 [gt2 ...]`
(relative to the raw root), then uncomment the commands. Each store is uploaded to HF.

In [ ]:
# ==== SETTINGS (nothing here needs to change between Kaggle accounts) ====
HF_REPO = "sujalprasad90/pedger-mtkd-store"   # the ONE private Hugging Face repo shared by every account
ACCOUNT = ""                                  # optional label, e.g. your name; sessions are told apart automatically
REPO_URL = ""                                 # optional GitHub URL of the code ("" = Kaggle dataset or HF copy)
# Once per Kaggle account, and for every imported notebook:
#   Add-ons -> Secrets -> HF_TOKEN = a WRITE token from the Hugging Face account that owns HF_REPO, ticked for this notebook
#   Settings -> Internet: On;  Settings -> Accelerator: as written in the title
# The preflight cell below checks all of this and stops with a plain-language fix if something is missing.

In [ ]:
# ---- setup: secrets, code, dependencies (no edits needed) ----
import glob, os, shutil, subprocess, sys, time
SESSION_START = time.time()
from kaggle_secrets import UserSecretsClient
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError("Secret HF_TOKEN is not available: Add-ons -> Secrets -> add HF_TOKEN and tick it for THIS "
                       "notebook, then run again.") from e
os.environ["PEDGER_REMOTE"] = "hf:" + HF_REPO
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.25,<2.0", "pyyaml"], check=True)
W, REPO, WS = "/kaggle/working", "/kaggle/working/pedger-mtkd", "/kaggle/working/ws"
if not os.path.exists(REPO):
    attached = sorted(glob.glob("/kaggle/input/*/pedger_kd/__init__.py") + glob.glob("/kaggle/input/*/*/pedger_kd/__init__.py"))
    if attached:                                   # code uploaded as a Kaggle Dataset (pedger-mtkd-code)
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), REPO)
        CODE_SOURCE = "kaggle dataset"
    elif REPO_URL:
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
        CODE_SOURCE = "git"
    else:                                          # copy pushed to Hugging Face by notebook 01
        from huggingface_hub import snapshot_download
        snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["code/*"], local_dir=f"{W}/hf_code",
                          token=os.environ["HF_TOKEN"])
        shutil.copytree(f"{W}/hf_code/code", REPO)
        CODE_SOURCE = "huggingface"
    print("code from:", CODE_SOURCE)
os.chdir(REPO)
sys.path.insert(0, REPO)
!python tools/fetch_upstream.py
# preflight: session type, GPUs, internet, Hugging Face account and repo access
if subprocess.run([sys.executable, "-m", "pedger_kd.preflight"]).returncode:
    raise RuntimeError("Preflight failed: fix the PROBLEM lines printed above, then run the notebook again.")

In [ ]:
RAW_NYUD = "/kaggle/input/nyud-hed"            # attach and inspect
!ls {RAW_NYUD} | head
# !python data/prepare_list.py --dataset nyud --raw-root {RAW_NYUD} --list {RAW_NYUD}/train_pair.lst --split trainval --out {WS}/data/nyud --max-dist 0.011
# !python data/prepare_list.py --dataset nyud --raw-root {RAW_NYUD} --list {RAW_NYUD}/test.lst --split test --eval-only --out {WS}/data/nyud --max-dist 0.011
# !python data/prepare_list.py --dataset nyud --raw-root {RAW_NYUD} --list my_val.lst --split val --eval-only --out {WS}/data/nyud --max-dist 0.011
# !python -m pedger_kd.sync push-dir --local {WS}/data/nyud --to data/nyud

In [ ]:
RAW_MC = "/kaggle/input/multicue"               # attach; check annotations per image
# !python data/make_multicue_splits.py --list multicue_boundary_all.lst --out {W}/lists/multicue_boundary
# for i in range(3):
#     !python data/prepare_list.py --dataset multicue_boundary --raw-root {RAW_MC} --list {W}/lists/multicue_boundary/split{i}_train.lst --split train --out {WS}/data/multicue_boundary_split{i}
#     !python data/prepare_list.py --dataset multicue_boundary --raw-root {RAW_MC} --list {W}/lists/multicue_boundary/split{i}_test.lst --split test --eval-only --out {WS}/data/multicue_boundary_split{i}
#     !python -m pedger_kd.sync push-dir --local {WS}/data/multicue_boundary_split{i} --to data/multicue_boundary_split{i}